# Week 2, Day 1 — May 25, 2026
## Drop Zero-Variance Columns & Null Imputation



In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, numpy as np, matplotlib.pyplot as plt, os, json, warnings
warnings.filterwarnings('ignore')
BASE_DIR = "/content/drive/MyDrive/Ocean_Plastic_Project"
DIRS = {
    "raw":            BASE_DIR + "/data/raw",
    "processed":      BASE_DIR + "/data/processed",
    "metadata":       BASE_DIR + "/data/metadata",
    "notebooks":      BASE_DIR + "/notebooks",
    "visualizations": BASE_DIR + "/visualizations"
}
LAT_MIN, LAT_MAX, LON_MIN, LON_MAX = 5, 30, 65, 95
YEAR_MIN, YEAR_MAX = 2015, 2026

In [ ]:
df_plastic = pd.read_csv(DIRS['processed'] + '/plastic_filtered.csv', parse_dates=['Date'])
df_species = pd.read_csv(DIRS['processed'] + '/species_filtered.csv')
df_climate = pd.read_csv(DIRS['processed'] + '/climate_flagged.csv',  parse_dates=['Date'])
df_temp    = pd.read_csv(DIRS['processed'] + '/global_temp_filtered.csv', parse_dates=['dt'])
print(f'Loaded: plastic={df_plastic.shape}, species={df_species.shape}, climate={df_climate.shape}')
print(f'Baseline nulls — species: {df_species.isnull().sum().sum()}, climate: {df_climate.isnull().sum().sum()}')

## Step 1: Drop High-Null & Unusable Columns

In [ ]:
# Columns to drop (identified in Week 1 schema audit)
species_drop = [
    'priority_group',   # 87% null — not recoverable
]
temp_drop = [
    'LandMaxTemperature', 'LandMaxTemperatureUncertainty',
    'LandMinTemperature', 'LandMinTemperatureUncertainty',
    'LandAverageTemperatureUncertainty',
    'LandAndOceanAverageTemperatureUncertainty',
]

df_species_clean = df_species.drop(columns=species_drop, errors='ignore')
df_temp_clean    = df_temp.drop(columns=temp_drop, errors='ignore')

print(f'species: {df_species.shape} → {df_species_clean.shape}  (dropped {len(species_drop)} col)')
print(f'temp:    {df_temp.shape} → {df_temp_clean.shape}  (dropped {len(temp_drop)} cols)')

## Step 2: Numeric Null Imputation — Median

In [ ]:
# year and month — impute with median (low null rate: 0.9%)
median_year  = df_species_clean['year'].median()
median_month = df_species_clean['month'].median()

df_species_clean = df_species_clean.assign(
    year  = df_species_clean['year'].fillna(median_year),
    month = df_species_clean['month'].fillna(median_month)
)

# Cast year to int now that nulls are gone
df_species_clean['year']  = df_species_clean['year'].astype(int)
df_species_clean['month'] = df_species_clean['month'].astype(int)

print(f'year  imputed with median: {median_year}')
print(f'month imputed with median: {median_month}')
print(f'year  dtype after cast: {df_species_clean["year"].dtype}')

# GlobalTemp — forward-fill the 12 LandAverageTemperature nulls (time series)
df_temp_clean = df_temp_clean.sort_values('dt')
df_temp_clean['LandAverageTemperature'] = df_temp_clean['LandAverageTemperature'].ffill()
print(f'GlobalTemp LandAverageTemperature nulls after ffill: {df_temp_clean["LandAverageTemperature"].isnull().sum()}')

## Step 3: Categorical Null Imputation — 'Unknown'

In [ ]:
# Categorical columns with 17.4% null rate — safe to fill with 'Unknown'
cat_fill_cols = ['india_role', 'migration_season', 'location_type', 'seasonal_density']
for col in cat_fill_cols:
    before = df_species_clean[col].isnull().sum()
    df_species_clean[col] = df_species_clean[col].fillna('Unknown')
    print(f'{col}: {before} nulls → filled with "Unknown" ✅')

# Climate — Bleaching Severity 30% null
df_climate_clean = df_climate.copy()
before = df_climate_clean['Bleaching Severity'].isnull().sum()
df_climate_clean['Bleaching Severity'] = df_climate_clean['Bleaching Severity'].fillna('Unknown')
print(f'\nBleaching Severity: {before} nulls → filled with "Unknown" ✅')

## Step 4: Verify — No Remaining Critical Nulls

In [ ]:
print('POST-IMPUTATION NULL CHECK:')
print()
for label, df in [('species_clean',df_species_clean), ('climate_clean',df_climate_clean), ('temp_clean',df_temp_clean)]:
    remaining = df.isnull().sum()
    remaining = remaining[remaining > 0]
    if len(remaining) == 0:
        print(f'  {label}: ✅ No nulls in key columns')
    else:
        print(f'  {label}: Remaining nulls:')
        for col, cnt in remaining.items():
            print(f'    {col}: {cnt} ({cnt/len(df)*100:.1f}%)')

In [ ]:
# Save cleaned versions
df_species_clean.to_csv(DIRS['processed'] + '/species_clean_v1.csv', index=False)
df_climate_clean.to_csv(DIRS['processed'] + '/climate_clean_v1.csv', index=False)
df_temp_clean.to_csv(   DIRS['processed'] + '/temp_clean_v1.csv',    index=False)
print('Cleaned datasets saved to /data/processed/ ✅')
print(f'  species_clean_v1.csv : {df_species_clean.shape}')
print(f'  climate_clean_v1.csv : {df_climate_clean.shape}')
print(f'  temp_clean_v1.csv    : {df_temp_clean.shape}')

## ✅ Day 1 Week 2 Complete — May 25, 2026

**Next: May 26 — Text Mismatch Cleaning**